In [1]:
#@title Import revelant modules and define some useful functions
import os
import sys
assert sys.version_info >= (3, 5)   # Python ≥3.5 is required
import urllib

import numpy as np   # numpy for numerical linear algebra
import pandas as pd  # pandas for managing dataframes

#import tensorflow as tf
#from tensorflow.keras import layers

import sklearn
from sklearn import tree
assert sklearn.__version__ >= "0.20"  # Scikit-Learn version ≥0.20 required
import sklearn.metrics as metrics

# adjust reporting in pandas: max 10 rows and two-digit precision
#pd.options.display.max_rows = 10
pd.options.display.float_format = "{:,.5f}".format

# import matplotlib and pyplot: critical packages for plotting 
import matplotlib as mpl
import matplotlib.pyplot as plt
# Make sure Matplotlib runs inline, for nice figures
%matplotlib inline  
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# install the latest version of seaborn for nicer graphics
# Uncomment below line to install the package, then commment out again. Same for all pip commands in the sequel.
#!pip install seaborn==0.11.0  &> /dev/null
import seaborn as sns

from IPython.core.display import display, HTML
import base64

# install graphviz for visualizing decision trees
# !pip install graphviz
import graphviz

# install pdpbox for partial dependency visualization
# !pip install pdpbox &> /dev/null
from pdpbox import info_plots, pdp

# Ignore useless some warnings
import warnings
warnings.simplefilter(action="ignore")

# pydotplus for saving a dot file to PDF
# !pip install pydotplus
import pydotplus

def append_dummies(data, columns):
  categories = {}  # the dictionary with categories
  data_with_dummies = data.copy() # the changed dataframe
  for col in columns:
    if col not in data.columns:
      print("WARNING. Column '{}' not among the columns in the dataframe. Skipping it.")
    elif (data[col].dtype!=int and data[col].dtype!=float):
      dummy_df = pd.get_dummies(data[col])
      categories[col] = list(dummy_df.columns)  # keep all the categories
      data_with_dummies = pd.merge( left=data_with_dummies, \
                                    right=dummy_df, \
                                   left_index=True, right_index=True,\
                                   how="inner", suffixes=("", "") )
  return data_with_dummies, categories

Choose activity for tests and load the dataset

In [7]:
for activity in ["work"]:
    local_csv = "./datasets/dataset_"+activity+".csv"   # name of local file where you want to store the downloaded file
    training_testing_local_csv = "./datasets/dataset_training_testing_"+activity+".csv" 
    evaluating_local_csv = "./datasets/dataset_evaluating_"+activity+".csv" 

    data = pd.read_csv(local_csv, index_col=None)

    data_training_testing = data.iloc[np.r_[31387500:32677830]]
    display(data_training_testing)
    # data_training_testing.to_csv(training_testing_local_csv)


    # data_evaluating = data.iloc[31387500:32677830]
    # data_evaluating.to_csv(evaluating_local_csv)






KeyboardInterrupt: 

In [8]:
data_training_testing = data.iloc[np.r_[31387500:32677830]]
display(data_training_testing)

,age_group,activity,t,lockdown,icu_capacity,theta_schooling,beta,alpha_1_2,nu_other_act,nu_work,...,I,Ia,Ims,Ips,Iss,ICU,H,R,Rq,D
31387500,age_group_0_9,work,0,0.00000,"3,200.00000",1.00000,0.02351,0.38739,0.10000,0.42728,...,"2,142.16634",0.00000,0.00000,0.00000,0.00000,3.56698,13.54089,"160,265.76119",302.05300,2.01298
31387501,age_group_10_19,work,0,0.00000,"3,200.00000",1.00000,0.02351,0.38739,0.10000,0.42728,...,"4,538.14371",0.00000,0.00000,0.00000,0.00000,7.56494,27.71278,"271,949.86816",506.35268,3.35926
31387502,age_group_20_29,work,0,0.00000,"3,200.00000",1.00000,0.02351,0.38739,0.10000,0.42728,...,"4,359.28358",0.00000,0.00000,0.00000,0.00000,11.37140,106.89078,"311,907.61249","1,743.13193",21.34783
31387503,age_group_30_39,work,0,0.00000,"3,200.00000",1.00000,0.02351,0.38739,0.10000,0.42728,...,"5,535.26200",0.00000,0.00000,0.00000,0.00000,43.34427,266.24901,"386,435.61230","4,680.42001",99.81696
31387504,age_group_40_49,work,0,1.00000,"3,200.00000",1.00000,0.02351,0.38739,0.10000,0.42728,...,"4,956.18765",0.00000,0.00000,0.00000,0.00000,70.84931,269.52457,"358,060.05897","5,639.52610",212.38933
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32677825,age_group_40_49,work,89,1.00000,"2,900.00000",0.50000,0.02351,0.38739,0.20000,0.32728,...,"1,945.91667",16.02883,8.01442,8.01442,0.55441,26.67801,112.10973,"386,207.02297","7,564.90261",238.88330
32677826,age_group_50_59,work,89,1.00000,"2,900.00000",0.50000,0.02351,0.38739,0.20000,0.32728,...,"1,444.25583",84.11703,42.05851,42.05851,6.10175,55.78523,166.68387,"305,068.89813","11,627.41385",782.01228
32677827,age_group_60_69,work,89,1.00000,"2,900.00000",0.50000,0.02351,0.38739,0.20000,0.32728,...,798.93549,2.12472,1.06236,1.06236,0.32477,63.53274,157.25535,"149,895.69599","9,672.33033","1,565.66660"
32677828,age_group_70_79,work,89,0.00000,"2,900.00000",0.50000,0.02351,0.38739,0.20000,0.32728,...,253.53758,0.00000,0.00000,0.00000,0.00000,25.84953,107.07968,"65,398.02169","6,293.34168","1,905.15613"


In [ ]:
print(pd.__version__)

1.1.3


In [3]:
# Read the data into a `pandas` DataFrame
data = pd.read_csv(local_csv, index_col=None)

In [6]:
display(data.iloc[32677830])

age_group                                       age_group_0_9
activity                                                 work
t                                                           0
lockdown                                              0.00000
icu_capacity                                      2,900.00000
theta_schooling                                       0.50000
beta                                                  0.02367
alpha_1_2                                             0.38739
nu_other_act                                          0.10000
nu_work                                               0.42728
chi                                           1,859,951.50000
vg_life                                         333,786.17206
xi+vg_life                                    2,193,737.67206
p_H                                                   0.00126
p_ICU                                                 0.00074
p_ss                                                  0.00200
p_D     

In [ ]:
list(data.columns)

['age_group',
 'activity',
 't',
 'lockdown',
 'icu_capacity',
 'theta_schooling',
 'beta',
 'alpha_1_2',
 'nu_other_act',
 'nu_work',
 'chi',
 'vg_life',
 'xi+vg_life',
 'p_H',
 'p_ICU',
 'p_ss',
 'p_D',
 'econ_gradient_normalized',
 'econ_gradient_not_normalized',
 'I_perc',
 'R_perc',
 'H_perc',
 'ICU_utilization',
 'contacts_received',
 'contacts_received_into_deaths',
 'infections_received',
 'deaths_received',
 'deaths_received_econ',
 'contacts_given',
 'contacts_given_into_deaths',
 'infections_given',
 'deaths_given',
 'deaths_given_econ',
 'econ_contacts_ratio',
 'econ_contacts_death_cost_ratio',
 'ICU_new_daily_admissions',
 'ICU_seven_day_avg_daily_admissions',
 'Infections_new_rate_over_seven_days',
 'Infections_new_rate_over_seven_days_elderly',
 'S',
 'E',
 'I',
 'Ia',
 'Ims',
 'Ips',
 'Iss',
 'ICU',
 'H',
 'R',
 'Rq',
 'D']

Count NaN's in the dataset:

In [4]:
data.isnull().sum().sum()

669060

Take closer look at rows with NaN's:

In [5]:
if data.isnull().sum().sum() > 0:
    # print the non-null counts for each feature
    data.isna().sum()
    
    data_NaN_rows = data[data.isna().any(axis=1)]
    print(data_NaN_rows.iloc[0])
    print(data_NaN_rows.iloc[359])

age_group                                      age_group_70_79
activity                                                  work
t                                                            0
lockdown                                               1.00000
icu_capacity                                       2,900.00000
theta_schooling                                        1.00000
beta                                                   0.05540
alpha_1_2                                              0.42417
nu_other_act                                           0.10000
nu_work                                                0.42728
chi                                                    0.00000
vg_life                                                0.00000
xi+vg_life                                             0.00000
p_H                                                    0.07324
p_ICU                                                  0.04954
p_ss                                                   

Are all the NaN's in the `econ_contacts_death_cost_ratio`?

In [ ]:
data.columns[data.isna().any()].tolist()

[]

#### <font color=red> This is due to division of 0/0, because of `chi+vg_life` being 0.  Will need to remember this when training the trees.</font>

#### <font color=green>  Agreed. </font>

#### <font color=blue> [DI] We need to figure out HOW to resolve it. Not sure that deleting the rows is the best option.</font>

#### <font color=red> Leaving this check and comment in for now until we have converged on a way forward, such as removing the `econ_to_contacts_death_cost_ratio` feature altogether.</font>

In [ ]:
# summary stats
data.describe()

,t,lockdown,icu_capacity,theta_schooling,beta,alpha_1_2,nu_other_act,nu_work,chi,vg_life,...,I,Ia,Ims,Ips,Iss,ICU,H,R,Rq,D
count,"12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000",...,"12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000"
mean,44.50000,0.98481,"2,918.75000",0.59375,0.02578,0.38322,0.10000,0.43528,"3,015,446.36938","275,818.93646",...,"3,285.40436",1.23740,0.61870,0.61870,0.05859,74.67677,326.10560,"332,981.31312","7,933.45393","2,244.80102"
std,25.98016,0.11621,72.62124,0.19516,0.00883,0.04062,0.00000,0.02294,"7,110,952.71882","203,172.79904",...,"3,565.54281",11.36510,5.68255,5.68255,0.53523,102.60852,329.23042,"314,019.50859","13,077.87615","7,258.66818"
min,0.00000,0.00000,"2,900.00000",0.50000,0.02316,0.24643,0.10000,0.42728,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,"28,987.94762",299.02020,1.31631
25%,22.00000,1.00000,"2,900.00000",0.50000,0.02351,0.38739,0.10000,0.42728,"371,990.30000","51,435.67968",...,620.08037,0.00000,0.00000,0.00000,0.00000,6.49172,28.30928,"153,987.65015","1,946.93720",23.45162
50%,44.50000,1.00000,"2,900.00000",0.50000,0.02351,0.38739,0.10000,0.42728,"557,985.45000","333,786.17206",...,"2,193.22621",0.00000,0.00000,0.00000,0.00000,37.43311,295.20861,"308,533.58512","6,234.02295",263.99480
75%,67.00000,1.00000,"2,900.00000",0.50000,0.02351,0.38739,0.10000,0.42728,"1,859,951.50000","438,746.52749",...,"4,744.17017",0.00000,0.00000,0.00000,0.00000,100.64028,472.57920,"403,938.84005","9,515.24283","1,758.83699"
max,89.00000,1.00000,"3,200.00000",1.00000,0.05997,0.46954,0.10000,0.51626,"29,908,020.12000","522,844.29491",...,"24,428.51854",190.06832,95.03416,95.03416,17.76352,889.76964,"2,558.18514","1,730,984.03299","123,334.32642","67,516.95327"


# Clean up the data 

Change $\chi$ and $\chi + \text{vg_life}$ values into multiples of GDP.

In [6]:
GDP_per_capita = 37199.03
data["chi_orig"] = data["chi"]
data["chi"] = data["chi_orig"]/GDP_per_capita

print(np.sort(data["chi"].unique()))

data["chi+vg_life_orig"] = data["xi+vg_life"]
data["chi+vg_life"] = data["chi+vg_life_orig"]/GDP_per_capita

print(np.sort(data["chi+vg_life"].unique()))


[0.00e+00 1.00e+00 3.00e+00 4.00e+00 5.00e+00 6.00e+00 7.00e+00 8.00e+00
 9.00e+00 1.00e+01 1.10e+01 1.50e+01 2.00e+01 2.50e+01 3.00e+01 3.50e+01
 4.00e+01 4.50e+01 5.00e+01 5.50e+01 6.00e+01 6.50e+01 7.00e+01 7.50e+01
 8.00e+01 8.50e+01 9.00e+01 9.50e+01 1.00e+02 1.05e+02 1.10e+02 1.15e+02
 1.18e+02 1.20e+02 1.25e+02 1.30e+02 1.35e+02 1.40e+02 1.45e+02 1.50e+02
 1.55e+02 1.60e+02 1.65e+02 1.70e+02 1.75e+02 1.80e+02 1.85e+02 1.90e+02
 1.95e+02 2.00e+02 2.10e+02 2.16e+02 2.20e+02 2.30e+02 2.40e+02 2.50e+02
 2.60e+02 2.70e+02 2.80e+02 2.90e+02 3.00e+02 3.10e+02 3.14e+02 3.20e+02
 3.30e+02 3.40e+02 3.50e+02 3.60e+02 3.70e+02 3.80e+02 3.90e+02 4.00e+02
 4.10e+02 4.12e+02 4.20e+02 4.30e+02 4.40e+02 4.50e+02 4.60e+02 4.70e+02
 4.80e+02 4.90e+02 5.00e+02 5.10e+02 5.20e+02 5.30e+02 5.40e+02 5.50e+02
 5.60e+02 5.70e+02 5.80e+02 5.90e+02 6.00e+02 6.08e+02 6.10e+02 6.20e+02
 6.30e+02 6.40e+02 6.50e+02 6.60e+02 6.70e+02 6.80e+02 6.90e+02 7.00e+02
 7.06e+02 7.10e+02 7.20e+02 7.30e+02 7.40e+02 7.50e

Add a new feature corresponding to the total population in the group without deaths. We add this as a proxy for the total population in the group that is working, which is what appears in the objective. This is easier to interpret, and DMs can compute it in practice.

In [7]:
alive_states = ['S', 'E', 'I', 'Ia', 'Ims', 'Ips', 'Iss', 'ICU', 'H', 'R', 'Rq']
data["Nalive"] = sum(data[s] for s in alive_states )

# Some Brief EDA

In [ ]:
# Summary stats
data.describe()

,t,lockdown,icu_capacity,theta_schooling,beta,alpha_1_2,nu_other_act,nu_work,chi,vg_life,...,Iss,ICU,H,R,Rq,D,chi_orig,chi+vg_life_orig,chi+vg_life,Nalive
count,"12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000",...,"12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000","12,960.00000"
mean,44.50000,0.98481,"2,918.75000",0.59375,0.02578,0.38322,0.10000,0.43528,81.06250,"275,818.93646",...,0.05859,74.67677,326.10560,"332,981.31312","7,933.45393","2,244.80102","3,015,446.36938","3,291,265.30584",88.47718,"1,362,000.75356"
std,25.98016,0.11621,72.62124,0.19516,0.00883,0.04062,0.00000,0.02294,191.15963,"203,172.79904",...,0.53523,102.60852,329.23042,"314,019.50859","13,077.87615","7,258.66818","7,110,952.71882","7,113,854.63414",191.23764,"408,165.56980"
min,0.00000,0.00000,"2,900.00000",0.50000,0.02316,0.24643,0.10000,0.42728,0.00000,0.00000,...,0.00000,0.00000,0.00000,"28,987.94762",299.02020,1.31631,0.00000,0.00000,0.00000,"478,713.02764"
25%,22.00000,1.00000,"2,900.00000",0.50000,0.02351,0.38739,0.10000,0.42728,10.00000,"51,435.67968",...,0.00000,6.49172,28.30928,"153,987.65015","1,946.93720",23.45162,"371,990.30000","557,985.45000",15.00000,"1,176,115.78617"
50%,44.50000,1.00000,"2,900.00000",0.50000,0.02351,0.38739,0.10000,0.42728,15.00000,"333,786.17206",...,0.00000,37.43311,295.20861,"308,533.58512","6,234.02295",263.99480,"557,985.45000","894,834.59491",24.05532,"1,546,105.13864"
75%,67.00000,1.00000,"2,900.00000",0.50000,0.02351,0.38739,0.10000,0.42728,50.00000,"438,746.52749",...,0.00000,100.64028,472.57920,"403,938.84005","9,515.24283","1,758.83699","1,859,951.50000","2,193,737.67206",58.97298,"1,651,894.09501"
max,89.00000,1.00000,"3,200.00000",1.00000,0.05997,0.46954,0.10000,0.51626,804.00000,"522,844.29491",...,17.76352,889.76964,"2,558.18514","1,730,984.03299","123,334.32642","67,516.95327","29,908,020.12000","30,430,864.41491",818.05532,"1,757,754.77249"


# Train a Decision Tree

In [8]:
from sklearn import tree
from sklearn.metrics import mean_squared_error

### Encode age group as numeric feature
Add another column with age encoded as a numeric feature

In [9]:
# ordinal encoding - two options:
#  1) replace with midpoint of closed interval (e.g., 5, 15, 25, etc; for 80+, will use 85)
#  2) replace age bracket with upper limit (for the 80-plus, will use 89)
use_midpoint = 1

data["age"] = data["age_group"]

unique_age_groups = data["age"].unique()
for v in unique_age_groups:
  all_splits = v.split("_")
  lower_lim = np.int(all_splits[-2])
  upper_lim = all_splits[-1]
  if upper_lim=="plus":
      upper_lim = 89
  else:
      upper_lim = np.int(upper_lim)

  if use_midpoint:
      data.loc[ data["age"]==v, "age" ] = (lower_lim + upper_lim + 1)/2
  else:
      data.loc[ data["age"]==v, "age" ] = upper_lim

# convert entire column for age into integer
data = data.astype({'age':int})

In [ ]:
data.head(9)

,age_group,activity,t,lockdown,icu_capacity,theta_schooling,beta,alpha_1_2,nu_other_act,nu_work,...,ICU,H,R,Rq,D,chi_orig,chi+vg_life_orig,chi+vg_life,Nalive,age
0,age_group_0_9,other,0,1.00000,"2,900.00000",1.00000,0.05997,0.46954,0.10000,0.42728,...,0.00000,0.00000,"1,553,054.75214","3,037.55556",747.81190,"4,389,485.54000","4,723,271.71206",126.97298,"1,594,108.18772",5
1,age_group_10_19,other,0,1.00000,"2,900.00000",1.00000,0.05997,0.46954,0.10000,0.42728,...,0.00000,0.00000,"1,542,008.26305","3,024.87920",733.56389,"4,389,485.54000","4,828,232.06749",129.79457,"1,545,375.43573",15
2,age_group_20_29,other,0,1.00000,"2,900.00000",1.00000,0.05997,0.46954,0.10000,0.42728,...,0.00000,0.00000,"1,639,888.71867","10,369.26804","1,314.08643","4,389,485.54000","4,912,329.83491",132.05532,"1,650,602.91241",25
3,age_group_30_39,other,0,1.00000,"2,900.00000",1.00000,0.05997,0.46954,0.10000,0.42728,...,0.00000,0.00000,"1,730,984.03299","22,755.56720","4,094.16696","4,389,485.54000","4,906,230.12862",131.89135,"1,753,753.83035",35
4,age_group_40_49,other,0,1.00000,"2,900.00000",1.00000,0.05997,0.46954,0.10000,0.42728,...,0.00000,0.00000,"1,637,511.24235","26,971.17545","7,293.03308","4,389,485.54000","4,793,113.76329",128.85050,"1,664,517.96349",45
5,age_group_50_59,other,0,1.00000,"2,900.00000",1.00000,0.05997,0.46954,0.10000,0.42728,...,0.00000,0.00000,"1,473,397.86402","46,740.47339","18,256.79285","4,389,485.54000","4,604,670.48210",123.78469,"1,520,379.20065",55
6,age_group_60_69,other,0,1.00000,"2,900.00000",1.00000,0.05997,0.46954,0.10000,0.42728,...,0.00000,0.00000,"1,069,498.65429","65,487.44857","35,767.46137","4,389,485.54000","4,440,921.21968",119.38272,"1,142,073.52851",65
7,age_group_70_79,other,0,1.00000,"2,900.00000",1.00000,0.05997,0.46954,0.10000,0.42728,...,0.00000,0.00000,"663,800.52862","64,779.32180","39,499.84349","4,389,485.54000","4,389,485.54000",118.00000,"753,462.14608",75
8,age_group_80_plus,other,0,1.00000,"2,900.00000",1.00000,0.05997,0.46954,0.10000,0.42728,...,0.00000,0.00001,"329,288.93801","123,334.32642","67,516.95327","4,389,485.54000","4,389,485.54000",118.00000,"478,713.02765",85


In [11]:
list(data.columns)

['age_group',
 'activity',
 't',
 'lockdown',
 'icu_capacity',
 'theta_schooling',
 'beta',
 'alpha_1_2',
 'nu_other_act',
 'nu_work',
 'chi',
 'vg_life',
 'xi+vg_life',
 'p_H',
 'p_ICU',
 'p_ss',
 'p_D',
 'econ_gradient_normalized',
 'econ_gradient_not_normalized',
 'I_perc',
 'R_perc',
 'H_perc',
 'ICU_utilization',
 'contacts_received',
 'contacts_received_into_deaths',
 'infections_received',
 'deaths_received',
 'deaths_received_econ',
 'contacts_given',
 'contacts_given_into_deaths',
 'infections_given',
 'deaths_given',
 'deaths_given_econ',
 'econ_contacts_ratio',
 'econ_contacts_death_cost_ratio',
 'ICU_new_daily_admissions',
 'ICU_seven_day_avg_daily_admissions',
 'Infections_new_rate_over_seven_days',
 'Infections_new_rate_over_seven_days_elderly',
 'S',
 'E',
 'I',
 'Ia',
 'Ims',
 'Ips',
 'Iss',
 'ICU',
 'H',
 'R',
 'Rq',
 'D',
 'chi_orig',
 'chi+vg_life_orig',
 'chi+vg_life',
 'Nalive',
 'age']

### Choose the variables to train the tree with; set filters; Train the tree, Display, Save to a File; Implement Cross-Validation

### Comments on features to keep in the trees
We remove the following features:
 - $\chi, v_g^{\mbox{life}}$ : we only retain $\chi + v_g^{\mbox{life}}$, since that is the only thing appearing in the model
 - we eliminate $p_{ss}$ since it is the sum of $p^H$ and $p^{ICU}$
 - 'econ_gradient_not_normalized': remove and only retain the other one

### Our plan will be to run a few different versions of trees:
 1. ALL: A tree with ALL the features (after the cleaning above). The point of this exercise is more to show that econ_to_contacts  is the most relevant feature, and also to see how good this is relative to ROLD.
 2. SIMPLE: A tree that only uses simple features, i.e. things that regulators are very likely to look at (see list below).
 3. SIMPLE_PLUS: A tree with the same simple features as #2 and the `econ_to_contacts_ratio`. 

In [10]:
# features for complete tree (#1 in list)
all_features = ['t', 'icu_capacity', 'theta_schooling', 'beta', 'alpha_1_2', 'nu_other_act', 'nu_work',
 'chi+vg_life', 'p_H', 'p_ICU', 'p_D', 'econ_gradient_normalized', 'I_perc', 'R_perc', 'H_perc', 'ICU_utilization',
 'contacts_received', 'contacts_received_into_deaths', 'infections_received', 'deaths_received', 'deaths_received_econ',
 'contacts_given', 'contacts_given_into_deaths', 'infections_given', 'deaths_given', 'deaths_given_econ',
 'econ_contacts_ratio', 'ICU_new_daily_admissions', 'ICU_seven_day_avg_daily_admissions',
 'Infections_new_rate_over_seven_days', 'Infections_new_rate_over_seven_days_elderly', 'S', 'E', 'I', 'Ia', 'Ims', 'Ips',
 'Iss', 'ICU', 'H', 'R', 'Rq', 'D', 'age', 'Nalive']

# features  for the simple tree
simple_features = ['t', 'icu_capacity', 'theta_schooling', 'beta', 'alpha_1_2', 'nu_other_act', 'nu_work',
 'chi+vg_life', 'p_H', 'p_ICU', 'p_D', 'econ_gradient_normalized', 'I_perc', 'R_perc', 'H_perc', 'ICU_utilization',
 'contacts_received', 'contacts_given', 'ICU_new_daily_admissions', 'ICU_seven_day_avg_daily_admissions',
 'Infections_new_rate_over_seven_days', 'Infections_new_rate_over_seven_days_elderly', 'S', 'E', 'I', 'Ia', 'Ims', 'Ips',
 'Iss', 'ICU', 'H', 'R', 'Rq', 'D', 'age', 'Nalive']

# features  for the simple tree and econ to contacts ratio
simple_plus_features = simple_features + ['econ_contacts_ratio']

In [11]:
#  Building the tree

#@markdown (Optional) Select other features and associated values to use for filtering:
#@param ['None', 'age', 'lockdown', 'econ_gradient', 'I_perc', 'R_perc', 'H_perc', 'ICU_utilization', \
# 'xi+cost_death', 'xi', 'cost_death', 'contacts_received', 'contacts_received(deaths)', 'infections_received', 
# 'deaths_received','deaths_econ_received', 'contacts_given', 'contacts_given(deaths)','infections_given', 
# 'deaths_given', 'deaths_econ_given', 'death_prob','t']
filter_feature_1 = "None" 
#filter_feature_1 = "xi" 
filter_feature_1_value = 1859951.5 #@param {type:"raw"}

filter_feature_2 = "None" 
#@param ['None', 'age', 'lockdown', 'econ_gradient', 'I_perc', 'R_perc', 'H_perc', 'ICU_utilization', \
# 'xi+cost_death', 'xi', 'cost_death', 'contacts_received', 'contacts_received(deaths)', 'infections_received', \
# 'deaths_received','deaths_econ_received', 'contacts_given', 'contacts_given(deaths)','infections_given', \
# 'deaths_given', 'deaths_econ_given', 'death_prob','t']
filter_feature_2_value = 0 #@param {type:"raw"}

#filter_feature_2 = "xi" #@param ['None', 'age', 'lockdown', 'econ_gradient', 'I_perc', 'R_perc', 'H_perc', \
# 'ICU_utilization', 'xi+cost_death', 'xi', 'cost_death', 'contacts_received', 'contacts_received(deaths)', \
# 'infections_received', 'deaths_received','deaths_econ_received', 'contacts_given', 'contacts_given(deaths)',\
# 'infections_given', 'deaths_given', 'deaths_econ_given', 'death_prob','t']
#filter_feature_2_value = 1859951.5 #@param {type:"raw"}

filter_feature_3 = 'None' #@param ['None', 'age', 'lockdown', 'econ_gradient', 'I_perc', 'R_perc', 'H_perc', \
# 'ICU_utilization', 'xi+cost_death', 'xi', 'cost_death', 'contacts_received', 'contacts_received(deaths)', \
# 'infections_received', 'deaths_received','deaths_econ_received', 'contacts_given', 'contacts_given(deaths)',\
# 'infections_given', 'deaths_given', 'deaths_econ_given', 'death_prob','t']
filter_feature_3_value_min = 0 #@param {type:"raw"}
filter_feature_3_value_max = 200

filter_feature_4 = 'None' #@param ['None', 'age', 'lockdown', 'econ_gradient', 'I_perc', 'R_perc', 'H_perc', \
# 'ICU_utilization', 'xi+cost_death', 'xi', 'cost_death', 'contacts_received', 'contacts_received(deaths)', \
# 'infections_received', 'deaths_received','deaths_econ_received', 'contacts_given', 'contacts_given(deaths)', \
# 'infections_given', 'deaths_given', 'deaths_econ_given', 'death_prob','t']
filter_feature_4_value_min = 10 #@param {type:"raw"}
filter_feature_4_value_max = 70 #@param {type:"raw"}

# CREATE A SUITABLE TRAINING SET
pydot_graph = {}  #dictionary with graphs for each selected activity

# SELECT THE FEATURES
#vars_to_use = all_features
#var_title = "ALL"
#vars_to_use = simple_features
#var_title = "SIMPLE"
vars_to_use = simple_plus_features
var_title = "SIMPLE_PLUS"

print("Building tree for {} activity, using {} features".format(activity,var_title))

# FILTER BASED ON CONDITIONS
filter = [True]*len(data)
if filter_feature_1!="None":
  title += "_{}_{}".format(filter_feature_1,filter_feature_1_value)
  filter = filter & (data[filter_feature_1] == filter_feature_1_value)
if filter_feature_2!="None":
  title += "_{}_{}".format(filter_feature_2,filter_feature_2_value)
  filter = filter & (data[filter_feature_2] == filter_feature_2_value)
if filter_feature_3!="None":
  filter = filter & (data[filter_feature_3] >= filter_feature_3_value_min) & \
(data[filter_feature_3] <= filter_feature_3_value_max)
  title += "_{}_min_{}_max_{}".format(filter_feature_3,filter_feature_3_value_min,filter_feature_3_value_max)
if filter_feature_4!="None":
  filter = filter & (data[filter_feature_4] >= filter_feature_4_value_min) & \
(data[filter_feature_4] <= filter_feature_4_value_max)
  title += "_{}_min_{}_max_{}".format(filter_feature_4,filter_feature_4_value_min,filter_feature_4_value_max)

# CREATE THE DATASET
train_set = data[filter]  # only rows for the correct activity

# remove all rows with missing data (for now...)
if( train_set.isna().sum().sum() > 0 ):
    train_set = train_set.dropna()
print("Applying the filter for '{}' results in {} unique records.".format(activity,len(train_set)))

# select the features
X_train = train_set.loc[:, vars_to_use ]
y_train = train_set.loc[:, "lockdown"]

# TRAIN THE TREE
# USE CROSS VALIDATION TO SEARCH FOR BEST max_depth
from sklearn.model_selection import GridSearchCV
param_grid = {'max_depth': np.arange(3,10,1), 'random_state' : [123]}

# Use random_state to set the seed in order to avoid instabilities across different runs due to randomization in the folding
dt_model = tree.DecisionTreeRegressor(criterion='mse',random_state=0)
grid_search = GridSearchCV(dt_model, param_grid, cv=10, scoring='neg_mean_squared_error', \
                          n_jobs=32, refit=True, return_train_score=True)

# fit using CV
grid_search.fit(X_train, y_train)

# get the best estimator
dt_model = grid_search.best_estimator_   
max_depth = dt_model.get_params()['max_depth']

print("Optimal depth: {}".format(max_depth))

# IF YOU WANT TO DISPLAY THE TREE UNCOMMENT THE NEXT SECTION
#dot_data = tree.export_graphviz(dt_model, max_depth=opt_max_depth, \
#                              feature_names=feature_names, \
#                              class_names=None, label='all', filled=True, impurity=False, node_ids=False, \
#                              rounded=True, rotate=True, proportion=False, precision=4)
#graph = graphviz.Source(dot_data)  
#display(graph)

# WE NEED TO CLEAN UP THE NAMES BELOW

# WRITE TREE TO DOT FILE
feature_names = list( X_train.columns )
if 'chi' in feature_names:
    feature_names[feature_names.index('chi')] = chr(967)
if 'chi+vg_life' in feature_names:
    feature_names[feature_names.index('chi+vg_life')] = chr(967)+' '+'+'+' '+'vg-life'    
if 't' in feature_names:
    feature_names[feature_names.index('t')] = 'time'
if 'econ_contacts_ratio' in feature_names:
    feature_names[feature_names.index('econ_contacts_ratio')] = 'econ-to-contacts-ratio'
if 'econ_contacts_death_cost_ratio' in feature_names:
    feature_names[feature_names.index('econ_contacts_death_cost_ratio')] = 'econ-to-contacts-deathcost-ratio'
if 'econ_gradient_trunc' in feature_names:
    feature_names[feature_names.index('econ_gradient_trunc')] = 'econ-gradient'
if 'R_perc' in feature_names:
    feature_names[feature_names.index('R_perc')] = 'R-perc'
if 'contacts_received' in feature_names:
    feature_names[feature_names.index('contacts_received')] = 'contacts-received'
if 'deaths_given_econ' in feature_names:
    feature_names[feature_names.index('deaths_given_econ')] = 'deaths-given-econ'

dot_data = tree.export_graphviz(dt_model, max_depth=max_depth, \
                              feature_names=feature_names, \
                              class_names=None, label='all', filled=True, impurity=False, node_ids=False, \
                              rounded=True, rotate=True, proportion=False, precision=4)

# WRITE TO A PDF FILE AND DOWNLOAD IT
title = activity

# adjust the file name to reflect the tree depth and which vars are used 
title += "_Dep_" + str(max_depth)
title += '_variables_' + var_title

# calculate MSE manually and check against the printed value
y_hat = dt_model.predict(X_train)
mse = sklearn.metrics.mean_squared_error(y_hat, y_train)              

# read in dot file and add a node with the correct MSE (for the prediction using the tree)
pydot_graph[activity] = pydotplus.graph_from_dot_data(dot_data)

# change any nodes with "age <=" into "age <"
for n in pydot_graph[activity].get_node_list():
    label = n.get_label()

    if label is not None:
        #print("Original label: {}".format(label))

        # change the precision for the logical condition
        if label.find('<=')>0:
            # not a leaf node
            logical_cond_thresh = (label.split("<= ")[1]).split('\\')[0]  # what comes after "<= " and before "\"
            logical_cond_thresh_val = np.float(logical_cond_thresh)  # the value just before the \
            label = label.replace(logical_cond_thresh, "{:.1f}".format(logical_cond_thresh_val))

        # change the formatting for samples (add ,)
        samples_str = (label.split("samples = ")[1]).split('\\')[0]  # what comes after "samples = " and before "\"
        samples_val = np.float(samples_str)  # the value just before the \
        label = label.replace(samples_str, "{:,.0f}".format(samples_val))

        node_value_str = label.split("value = ")[1][0:-1]  # what comes after "value =" in the node label
        node_value = np.float(node_value_str)
        #print("Node value: {}".format(node_value_str))

        offset = 0.0
        n.set_fillcolor("0.0 0.0 "+str(offset + node_value*(1-offset)))

        if node_value<0.5:
            # for nodes that are sufficiently dark, make the fonts white
            n.set_fontcolor('white')

        label = label.replace("value","activity level")
        label = label.replace(node_value_str,"{:.2f}%".format(node_value*100))

        #in case we need to change the label for age...
        if label.find("age <=")>0:
             label.replace("age <=","age <")

        # print("New label {}\n".format(label))
        n.set_label(label)

# calculate MSE
mse_string = "RMSE: {:.5f}".format(np.sqrt(mse))
print(mse_string)

show_rmse = True
if show_rmse:
    title_node = pydotplus.Node(label='Decision tree for "'+activity+'"\n'+mse_string, \
                                fontcolor="black", fillcolor="white", fontsize=18, style="bold")
else:
    title_node = pydotplus.Node(label='Decision tree for "'+activity+'"', \
                                fontcolor="black", fillcolor="white", fontsize=18, style="bold")

pydot_graph[activity].add_node(title_node)

file_name = title + ".pdf"

pydot_graph[activity].write_pdf(file_name)

# DOWNLOAD THE PDF -- uncomment if in COLAB
# files.download(file_name)

Building tree for work activity, using SIMPLE_PLUS features
Applying the filter for 'work' results in 34973370 unique records.
